In [24]:
import pandas as pd
import numpy as np

# ============================================================
# IMPORT DATA
# ============================================================
data = pd.read_csv("heavy_metals.csv")
samples = data["sample_location"]
metals = ["Fe","Pb","Cu","Cr","Mn","Zn","Ni","Co","Cd"]

# ============================================================
# STANDARDS (WHO / BIS)
# desirable and permissible limits
# ============================================================

desirable = {
"Fe":0.3,"Pb":0.01,"Cu":0.05,"Cr":0.05,
"Mn":0.1,"Zn":5,"Ni":0.02,"Co":0.05,"Cd":0.003
}

permissible = {
"Fe":1.0,"Pb":0.01,"Cu":1.5,"Cr":0.05,
"Mn":0.4,"Zn":15,"Ni":0.07,"Co":0.1,"Cd":0.005
}

background = {
"Fe":0.05,"Pb":0.002,"Cu":0.01,"Cr":0.002,
"Mn":0.005,"Zn":0.02,"Ni":0.002,"Co":0.001,"Cd":0.0001
}

toxicity = {
"Cd":30,"Pb":5,"Ni":5,"Cu":5,"Cr":2,"Zn":1,"Mn":1,"Fe":1,"Co":5
}
toxic_response_factors = {
    'Cd': 30,  # Highest toxicity - Hakanson (1980)
    'Pb': 5,   # Moderate toxicity - Hakanson (1980)
    'Cu': 5,   # Moderate toxicity - Hakanson (1980)
    'Cr': 2,   # Low toxicity - Hakanson (1980)
    'Ni': 5,   # Moderate toxicity - recent studies [citation:1]
    'Co': 5,   # Moderate toxicity - recent studies [citation:1]
    'Zn': 1,   # Lowest toxicity - Hakanson (1980)
    'Fe': 1,   # Essential element - default
    'Mn': 1    # Essential element - default
}


In [12]:
import pandas as pd

def calculate_cf():

    data = pd.read_csv("heavy_metals.csv")

    result = data.copy()

    for metal in metals:
        result[metal] = data[metal] / background[metal]

    result.to_csv("outputs/CF_results.csv", index=False)

    return result


In [13]:
calculate_cf()

,sample_location,Fe,Pb,Cu,Cr,Mn,Zn,Ni,Co,Cd
0,LP1,18.980000,6.000000,1.900000,5.500000,5.400000,0.550000,1.500000,3.000000,20.00000
1,LP2,18.200000,6.000000,1.600000,6.000000,6.000000,0.650000,1.500000,3.000000,20.00000
2,LP3,16.780000,7.000000,1.800000,6.000000,6.200000,0.550000,1.500000,3.000000,20.00000
3,LP4,30.480000,23.500000,2.600000,7.500000,4.400000,1.050000,2.000000,4.000000,30.00000
4,LP5,16.300000,8.000000,2.100000,6.000000,5.800000,0.750000,1.500000,4.000000,20.00000
5,LP6,34.060000,14.500000,2.100000,7.500000,6.200000,0.950000,2.000000,4.000000,30.00000
6,LP7,15.700000,7.500000,2.200000,5.000000,6.400000,0.600000,1.500000,3.000000,20.00000
7,AJ1,15.840000,5.500000,2.700000,4.500000,4.000000,0.500000,1.500000,3.000000,20.00000
8,AJ2,32.240000,14.500000,2.100000,7.500000,20.200000,0.950000,1.500000,4.000000,20.00000
9,AJ3,16.380000,7.500000,2.600000,6.000000,4.800000,0.650000,2.000000,3.000000,20.00000


In [ ]:
import pandas as pd

def calculate_cd():

    cf = pd.read_csv("outputs/CF_results.csv")

    metals = cf.columns[1:]

    cd = cf[metals].sum(axis=1)

    result = pd.DataFrame({
        "Sample": cf.sample_location,
        "Cd": cd
    })

    def classify(x):

        if x < 8:
            return "Low contamination"

        elif x < 16:
            return "Moderate contamination"

        elif x < 32:
            return "Considerable contamination"

        return "Very high contamination"

    result["Classification"] = result.Cd.apply(classify)

    result.to_csv("outputs/CD_results.csv", index=False)

    return result

In [18]:
calculate_cd()

,Sample,Cd,Classification
0,LP1,62.830000,Very high contamination
1,LP2,62.950000,Very high contamination
2,LP3,62.830000,Very high contamination
3,LP4,105.530000,Very high contamination
4,LP5,64.450000,Very high contamination
5,LP6,101.310000,Very high contamination
6,LP7,61.900000,Very high contamination
7,AJ1,57.540000,Very high contamination
8,AJ2,102.990000,Very high contamination
9,AJ3,62.930000,Very high contamination


In [27]:
import pandas as pd

toxic_response_factors = {
    'Cd': 30,  # Highest toxicity - Hakanson (1980)
    'Pb': 5,   # Moderate toxicity - Hakanson (1980)
    'Cu': 5,   # Moderate toxicity - Hakanson (1980)
    'Cr': 2,   # Low toxicity - Hakanson (1980)
    'Ni': 5,   # Moderate toxicity - recent studies
    'Co': 5,   # Moderate toxicity - recent studies
    'Zn': 1,   # Lowest toxicity - Hakanson (1980)
    'Fe': 1,   # Essential element - default
    'Mn': 1    # Essential element - default
}

def calculate_mcd():
    # Read CF results
    cf = pd.read_csv("outputs/CF_results.csv")
    
    # Get metals list (excluding sample_location column)
    metals = cf.columns[1:]
    
    # Initialize lists to store results for each sample
    mcd_values = []
    
    # Calculate MCD for each sample (row)
    for idx, row in cf.iterrows():
        weighted_sum = 0
        total_tr = 0
        
        for metal in metals:
            cf_value = row[metal]
            
            if metal in toxic_response_factors:
                tr = toxic_response_factors[metal]
                weighted_sum += cf_value * tr
                total_tr += tr
            else:
                # If metal not in dictionary, use default Tr = 1
                print(f"Warning: {metal} not found in toxic response factors. Using default Tr=1")
                weighted_sum += cf_value * 1
                total_tr += 1
        
        # Calculate MCD for this sample = Σ(CF × Tr) / Σ Tr
        if total_tr > 0:
            mcd = weighted_sum / total_tr
        else:
            mcd = 0
            print(f"Warning: No valid Tr values for sample {row['sample_location']}")
        
        mcd_values.append(mcd)
    
    # Create result dataframe
    result = pd.DataFrame({
        "Sample": cf["sample_location"],
        "mCd": mcd_values
    })
    
    # Add classification
    def classify_mcd(value):
        if value < 1.0:
            return "Low Contamination"
        elif 1.0 <= value < 2.0:
            return "Moderate Contamination"
        elif 2.0 <= value < 3.5:
            return "Considerable Contamination"
        else:
            return "High Contamination"
    
    result["Classification"] = result["mCd"].apply(classify_mcd)
    
    # Save results
    result.to_csv("outputs/mCD_results.csv", index=False)
    print(f"✓ MCD calculations complete. Processed {len(result)} samples")
    
    return result

In [28]:
calculate_mcd()

✓ MCD calculations complete. Processed 18 samples


,Sample,mCd,Classification
0,LP1,12.689636,High Contamination
1,LP2,12.679091,High Contamination
2,LP3,12.764182,High Contamination
3,LP4,20.207818,High Contamination
4,LP5,12.960909,High Contamination
5,LP6,19.440182,High Contamination
6,LP7,12.794545,High Contamination
7,AJ1,12.597091,High Contamination
8,AJ2,14.161636,High Contamination
9,AJ3,12.896909,High Contamination


In [30]:
import pandas as pd
import numpy as np

def calculate_pli():

    cf = pd.read_csv("outputs/CF_results.csv")

    metals = cf.columns[1:]

    pli = (cf[metals].prod(axis=1)) ** (1/len(metals))

    result = pd.DataFrame({
        "Sample": cf.sample_location,
        "PLI": pli
    })

    result["Interpretation"] = result.PLI.apply(
        lambda x: "Unpolluted" if x < 1 else "Polluted"
    )

    result.to_csv("outputs/PLI_results.csv", index=False)

    return result

def classify_pli(value):
    if value < 1.0:
        return "Unpolluted"
    elif 1.0 <= value < 2.0:
        return "Slightly Polluted"
    elif 2.0 <= value < 3.0:
        return "Moderately Polluted"
    elif 3.0 <= value < 4.0:
        return "Highly Polluted"
    else:
        return "Extremely Polluted"


if __name__ == "__main__":
    calculate_pli()

In [31]:
calculate_pli()

,Sample,PLI,Interpretation
0,LP1,4.086917,Polluted
1,LP2,4.153578,Polluted
2,LP3,4.179705,Polluted
3,LP4,6.293404,Polluted
4,LP5,4.563075,Polluted
5,LP6,6.058397,Polluted
6,LP7,4.245031,Polluted
7,AJ1,3.860672,Polluted
8,AJ2,6.357049,Polluted
9,AJ3,4.473510,Polluted


In [ ]:
import pandas as pd
import numpy as np

def calculate_npi():

    data = pd.read_csv("heavy_metals.csv")

    perm = dict(zip(limits.Metal, limits.Permissible))

    metals = data.columns[1:]

    ratios = data.copy()

    for metal in metals:
        ratios[metal] = data[metal] / perm[metal]

    mean_ratio = ratios[metals].mean(axis=1)
    max_ratio = ratios[metals].max(axis=1)

    npi = np.sqrt((mean_ratio**2 + max_ratio**2)/2)

    result = pd.DataFrame({
        "Sample": data.Sample,
        "NPI": npi
    })

    result.to_csv("outputs/NPI_results.csv", index=False)

    return result


if __name__ == "__main__":
    calculate_npi()

In [ ]:
import pandas as pd

def calculate_hei():

    data = pd.read_csv("data/water_concentration.csv")
    limits = pd.read_csv("data/standards_limits.csv")

    perm = dict(zip(limits.Metal, limits.Permissible))

    metals = data.columns[1:]

    ratios = data.copy()

    for metal in metals:
        ratios[metal] = data[metal] / perm[metal]

    hei = ratios[metals].sum(axis=1)

    result = pd.DataFrame({
        "Sample": data.Sample,
        "HEI": hei
    })

    result.to_csv("outputs/HEI_results.csv", index=False)

    return result


if __name__ == "__main__":
    calculate_hei()

In [ ]:
import pandas as pd
import numpy as np

def calculate_igeo():

    data = pd.read_csv("data/water_concentration.csv")
    background = pd.read_csv("data/background_values.csv")

    bg = dict(zip(background.Metal, background.Background))

    metals = data.columns[1:]

    result = pd.DataFrame()

    for metal in metals:
        result[metal] = np.log2(data[metal] / (1.5 * bg[metal]))

    result.insert(0,"Sample",data.Sample)

    result.to_csv("outputs/IGEO_results.csv", index=False)

    return result


if __name__ == "__main__":
    calculate_igeo()

In [ ]:
import pandas as pd

def calculate_eri():

    cf = pd.read_csv("outputs/CF_results.csv")
    tox = pd.read_csv("data/toxicity_factors.csv")

    toxicity = dict(zip(tox.Metal, tox.Toxicity))

    metals = cf.columns[1:]

    result = pd.DataFrame()

    for metal in metals:
        result[metal] = cf[metal] * toxicity.get(metal,1)

    result.insert(0,"Sample",cf.Sample)

    result.to_csv("outputs/ERI_results.csv", index=False)

    return result


if __name__ == "__main__":
    calculate_eri()

In [ ]:
import pandas as pd

def calculate_mi():

    data = pd.read_csv("data/water_concentration.csv")
    limits = pd.read_csv("data/standards_limits.csv")

    perm = dict(zip(limits.Metal, limits.Permissible))

    metals = data.columns[1:]

    ratios = data.copy()

    for metal in metals:
        ratios[metal] = data[metal] / perm[metal]

    mi = ratios[metals].sum(axis=1)

    result = pd.DataFrame({
        "Sample": data.Sample,
        "MI": mi
    })

    result.to_csv("outputs/MI_results.csv", index=False)

    return result


if __name__ == "__main__":
    calculate_mi()

In [37]:
import pandas as pd
import numpy as np

def classify_npi(value):

    if value <= 0.7:
        return "Clean", "Water is safe and free from heavy metal pollution."
    elif value <= 1:
        return "Warning Level", "Slight contamination begins."
    elif value <= 2:
        return "Light Pollution", "Water quality deterioration detected."
    elif value <= 3:
        return "Moderate Pollution", "Heavy metal pollution evident."
    else:
        return "Heavy Pollution", "Serious contamination requiring intervention."


def calculate_npi():

    data = pd.read_csv("heavy_metals.csv")
    limits = pd.read_csv("standard_limits.csv")

    perm = dict(zip(limits.Metal, limits.Permissible))
    metals = data.columns[1:]

    ratios = data.copy()

    for metal in metals:
        ratios[metal] = data[metal] / perm.get(metal,1)

    mean_ratio = ratios[metals].mean(axis=1)
    max_ratio = ratios[metals].max(axis=1)

    npi = np.sqrt((mean_ratio**2 + max_ratio**2)/2)

    classification = []
    interpretation = []

    for val in npi:
        c,i = classify_npi(val)
        classification.append(c)
        interpretation.append(i)

    result = pd.DataFrame({
        "Sample":data.sample_location,
        "NPI":npi.round(3),
        "Classification":classification,
        "Interpretation":interpretation
    })

    result.to_csv("outputs/NPI_results.csv",index=False)

    return result

In [38]:
calculate_npi()

,Sample,NPI,Classification,Interpretation
0,LP1,0.879,Warning Level,Slight contamination begins.
1,LP2,0.879,Warning Level,Slight contamination begins.
2,LP3,1.018,Light Pollution,Water quality deterioration detected.
3,LP4,3.372,Heavy Pollution,Serious contamination requiring intervention.
4,LP5,1.159,Light Pollution,Water quality deterioration detected.
5,LP6,2.099,Moderate Pollution,Heavy metal pollution evident.
6,LP7,1.087,Light Pollution,Water quality deterioration detected.
7,AJ1,0.804,Warning Level,Slight contamination begins.
8,AJ2,2.097,Moderate Pollution,Heavy metal pollution evident.
9,AJ3,1.089,Light Pollution,Water quality deterioration detected.


In [39]:
import pandas as pd

def classify_hei(value):

    if value < 10:
        return "Low Pollution","Water quality is acceptable."
    elif value < 20:
        return "Medium Pollution","Moderate heavy metal contamination."
    else:
        return "High Pollution","Serious heavy metal pollution."


def calculate_hei():

    data = pd.read_csv("heavy_metals.csv")
    limits = pd.read_csv("standard_limits.csv")

    perm = dict(zip(limits.Metal,limits.Permissible))
    metals = data.columns[1:]

    ratios = data.copy()

    for metal in metals:
        ratios[metal] = data[metal] / perm.get(metal,1)

    hei = ratios[metals].sum(axis=1)

    classification=[]
    interpretation=[]

    for val in hei:
        c,i = classify_hei(val)
        classification.append(c)
        interpretation.append(i)

    result = pd.DataFrame({
        "Sample":data.sample_location,
        "HEI":hei.round(3),
        "Classification":classification,
        "Interpretation":interpretation
    })

    result.to_csv("outputs/HEI_results.csv",index=False)

    return result

In [40]:
calculate_hei()

,Sample,HEI,Classification,Interpretation
0,LP1,2.923,Low Pollution,Water quality is acceptable.
1,LP2,2.909,Low Pollution,Water quality is acceptable.
2,LP3,3.042,Low Pollution,Water quality is acceptable.
3,LP4,7.295,Low Pollution,Water quality is acceptable.
4,LP5,3.225,Low Pollution,Water quality is acceptable.
5,LP6,5.693,Low Pollution,Water quality is acceptable.
6,LP7,3.053,Low Pollution,Water quality is acceptable.
7,AJ1,2.614,Low Pollution,Water quality is acceptable.
8,AJ2,5.563,Low Pollution,Water quality is acceptable.
9,AJ3,3.124,Low Pollution,Water quality is acceptable.


In [43]:
import pandas as pd
import numpy as np

def classify_igeo(val):

    if val <= 0:
        return "Unpolluted","Background concentration."
    elif val <= 1:
        return "Unpolluted to Moderately Polluted","Minor contamination."
    elif val <= 2:
        return "Moderately Polluted","Moderate enrichment."
    elif val <= 3:
        return "Moderately to Heavily Polluted","Significant contamination."
    elif val <= 4:
        return "Heavily Polluted","Serious anthropogenic impact."
    elif val <= 5:
        return "Heavily to Extremely Polluted","Very high pollution."
    else:
        return "Extremely Polluted","Severe contamination."


def calculate_igeo():

    data = pd.read_csv("heavy_metals.csv")
    background = pd.read_csv("standard_limits.csv")

    bg = dict(zip(background.Metal,background.Background))
    metals = data.columns[1:]

    results = []

    for i,row in data.iterrows():

        for metal in metals:

            val = np.log2(row[metal] / (1.5 * bg.get(metal,1)))

            c,i2 = classify_igeo(val)

            results.append([
                row["sample_location"],
                metal,
                val,
                c,
                i2
            ])

    result = pd.DataFrame(results,columns=[
        "Sample","Metal","Igeo","Classification","Interpretation"
    ])

    result.to_csv("outputs/IGEO_results.csv",index=False)

    return result

In [44]:
calculate_igeo()

,Sample,Metal,Igeo,Classification,Interpretation
0,LP1,Fe,3.661446,Heavily Polluted,Serious anthropogenic impact.
1,LP1,Pb,2.000000,Moderately Polluted,Moderate enrichment.
2,LP1,Cu,0.341037,Unpolluted to Moderately Polluted,Minor contamination.
3,LP1,Cr,1.874469,Moderately Polluted,Moderate enrichment.
4,LP1,Mn,1.847997,Moderately Polluted,Moderate enrichment.
...,...,...,...,...,...
157,Average,Mn,2.855833,Moderately to Heavily Polluted,Significant contamination.
158,Average,Zn,-0.645625,Unpolluted,Background concentration.
159,Average,Ni,0.281771,Unpolluted to Moderately Polluted,Minor contamination.
160,Average,Co,1.185556,Moderately Polluted,Moderate enrichment.


In [47]:
import pandas as pd

def classify_eri(val):

    if val < 40:
        return "Low Risk","Minimal ecological threat."
    elif val < 80:
        return "Moderate Risk","Moderate ecological risk."
    elif val < 160:
        return "Considerable Risk","Significant ecological threat."
    elif val < 320:
        return "High Risk","Serious ecological impact."
    else:
        return "Very High Risk","Severe ecological damage."


def calculate_eri():

    cf = pd.read_csv("outputs/CF_results.csv")
    tox = pd.read_csv("standard_limits.csv")

    toxicity = dict(zip(tox.Metal,tox.Toxicity))

    metals = cf.columns[1:]

    results=[]

    for i,row in cf.iterrows():

        for metal in metals:

            eri = row[metal] * toxicity.get(metal,1)

            c,i2 = classify_eri(eri)

            results.append([
                row["sample_location"],
                metal,
                eri,
                c,
                i2
            ])

    result = pd.DataFrame(results,columns=[
        "Sample","Metal","ERI","Classification","Interpretation"
    ])

    result.to_csv("outputs/ERI_results.csv",index=False)

    return result

In [ ]:
calculate_eri()

,Sample,Metal,ERI,Classification,Interpretation
0,LP1,Fe,18.980000,Low Risk,Minimal ecological threat.
1,LP1,Pb,30.000000,Low Risk,Minimal ecological threat.
2,LP1,Cu,9.500000,Low Risk,Minimal ecological threat.
3,LP1,Cr,11.000000,Low Risk,Minimal ecological threat.
4,LP1,Mn,5.400000,Low Risk,Minimal ecological threat.
...,...,...,...,...,...
157,Average,Mn,10.858824,Low Risk,Minimal ecological threat.
158,Average,Zn,0.958824,Low Risk,Minimal ecological threat.
159,Average,Ni,9.117648,Low Risk,Minimal ecological threat.
160,Average,Co,17.058825,Low Risk,Minimal ecological threat.


: 